In [53]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Using device:", DEVICE)

Using device: cuda


## Load Data

In [54]:
TRAIN_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_train.csv"
TRAIN_LABELS_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_train_labels.csv"
TEST_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
train_labels_df = pd.read_csv(TRAIN_LABELS_PATH)
test_df = pd.read_csv(TEST_PATH)

ID_COL = "sample_index"
TIME_COL = "time"
TARGET_COL = "label"
STATIC_COL = "is_pirate"

train_df = train_df.merge(train_labels_df[[ID_COL, TARGET_COL]], on=ID_COL, how="left")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Classes:", train_labels_df[TARGET_COL].unique())

Train shape: (105760, 41)
Test shape: (211840, 40)
Classes: ['no_pain' 'low_pain' 'high_pain']


## Preprocessing: Create Static Features and Drop Columns

In [55]:
def check_pirate(row):
    return 0 if row.get("n_legs", "two") == "two" else 1

if "n_legs" in train_df.columns:
    train_df["is_pirate"] = train_df.apply(check_pirate, axis=1)
    test_df["is_pirate"] = test_df.apply(check_pirate, axis=1)
    
    for col in ["n_legs", "n_hands", "n_eyes"]:
        if col in train_df.columns:
            train_df.drop(columns=col, inplace=True)
        if col in test_df.columns:
            test_df.drop(columns=col, inplace=True)

TO_DROP = ["joint_11", "joint_30"]
if all(col in train_df.columns for col in TO_DROP):
    train_df.drop(columns=TO_DROP, inplace=True)
if all(col in test_df.columns for col in TO_DROP):
    test_df.drop(columns=TO_DROP, inplace=True)

## Outlier Removal

In [56]:
JOINT_COLS = [c for c in train_df.columns if c.startswith("joint_")]
print("Number of joint features:", len(JOINT_COLS))

def compute_global_thresholds(df, joint_cols, q=0.999, multiplier=10.0):
    thresholds = {}
    for c in joint_cols:
        q_high = df[c].quantile(q)
        thresholds[c] = float(q_high * multiplier)
    return thresholds

def correct_glitches(df, joint_cols, thresholds, id_col="sample_index", time_col="time",
                     ratio_thresh=1e4, verbose=True, labels_df=None, label_col="label",
                     exclude_labels=None):
    import math
    eps = 1e-12
    total_cells = 0
    total_segments = 0
    
    working_df = df
    if labels_df is not None:
        if label_col not in df.columns:
            working_df = df.merge(labels_df[[id_col, label_col]], on=id_col, how="left")
    
    exclude_mask = None
    if exclude_labels is not None and label_col in working_df.columns:
        exclude_mask = working_df[label_col].isin(exclude_labels)

    for sid in df[id_col].unique():
        sub_mask = df[id_col] == sid
        sub_idx_sorted = df.loc[sub_mask].sort_values(time_col).index
        times = df.loc[sub_idx_sorted, time_col].to_numpy()
        nT = len(sub_idx_sorted)
        
        if exclude_mask is not None:
            subject_exclude_mask = exclude_mask.loc[sub_idx_sorted].to_numpy()
        else:
            subject_exclude_mask = None

        for col in joint_cols:
            vals_orig = df.loc[sub_idx_sorted, col].to_numpy(dtype=float)
            vals_new = vals_orig.copy()
            cand = vals_orig > thresholds[col]
            if not np.any(cand):
                continue

            cand_idx = np.where(cand)[0]
            runs = []
            start = cand_idx[0]
            prev = cand_idx[0]
            for k in cand_idx[1:]:
                if k == prev + 1:
                    prev = k
                else:
                    runs.append((start, prev))
                    start = k
                    prev = k
            runs.append((start, prev))

            for a, b in runs:
                if subject_exclude_mask is not None:
                    if np.any(subject_exclude_mask[a:b + 1]):
                        continue
                
                neighbour_idxs = []
                if a - 1 >= 0:
                    neighbour_idxs.append(a - 1)
                if a - 2 >= 0:
                    neighbour_idxs.append(a - 2)
                if b + 1 < nT:
                    neighbour_idxs.append(b + 1)
                if b + 2 < nT:
                    neighbour_idxs.append(b + 2)
                neighbour_idxs = sorted(set(neighbour_idxs))

                if not neighbour_idxs:
                    continue

                neighbour_vals = np.abs(vals_orig[neighbour_idxs])
                local_mean = neighbour_vals.mean()
                seg_vals = np.abs(vals_orig[a:b + 1])
                seg_max = seg_vals.max()

                if local_mean < eps:
                    ratio = math.inf if seg_max > 0 else 1.0
                else:
                    ratio = float(seg_max / (local_mean + eps))

                if seg_max > thresholds[col] and ratio >= ratio_thresh:
                    left_idx = a - 1 if a - 1 >= 0 else None
                    right_idx = b + 1 if b + 1 < nT else None
                    left_val = vals_orig[left_idx] if left_idx is not None else None
                    right_val = vals_orig[right_idx] if right_idx is not None else None

                    if (left_val is not None) and (right_val is not None):
                        length = right_idx - left_idx + 1
                        interp = np.linspace(left_val, right_val, length)
                        vals_new[a:b + 1] = interp[1:-1]
                    elif left_val is not None:
                        vals_new[a:b + 1] = left_val
                    elif right_val is not None:
                        vals_new[a:b + 1] = right_val
                    
                    total_segments += 1
                    total_cells += (b - a + 1)

            if not np.allclose(vals_new, vals_orig):
                df.loc[sub_idx_sorted, col] = vals_new

    if verbose:
        print(f"Outlier correction: {total_cells} cells in {total_segments} segments")
    
    return {"total_cells_corrected": total_cells, "total_segments": total_segments}

TIERA_QUANTILE = 0.999
TIERA_MULTIPLIER = 10.0
RATIO_THRESH = 1e4

global_thresholds = compute_global_thresholds(train_df, JOINT_COLS, q=TIERA_QUANTILE, multiplier=TIERA_MULTIPLIER)
train_summary = correct_glitches(train_df, JOINT_COLS, global_thresholds, id_col=ID_COL, time_col=TIME_COL,
                                 ratio_thresh=RATIO_THRESH, verbose=True, labels_df=train_labels_df,
                                 label_col="label", exclude_labels=['high_pain'])

Number of joint features: 29
Outlier correction: 7 cells in 4 segments


## Add Fourier Features

In [57]:
def add_fourier_features_df(df, time_col='time', periods=[4.0, 7.5]):
    for period in periods:
        freq = 2 * np.pi / period
        df[f'sin_p{period}'] = np.sin(freq * df[time_col]).astype(np.float32)
        df[f'cos_p{period}'] = np.cos(freq * df[time_col]).astype(np.float32)
    print(f"Added Fourier features for periods {periods}: {len(periods)*2} new features")
    return df

train_df = add_fourier_features_df(train_df, time_col=TIME_COL, periods=[4.0, 7.5])
test_df = add_fourier_features_df(test_df, time_col=TIME_COL, periods=[4.0, 7.5])

Added Fourier features for periods [4.0, 7.5]: 4 new features
Added Fourier features for periods [4.0, 7.5]: 4 new features


## Feature Scaling and Windowing

In [58]:
all_feature_cols = [c for c in train_df.columns if c not in [ID_COL, TIME_COL, TARGET_COL]]
test_feature_cols = [c for c in test_df.columns if c not in [ID_COL, TIME_COL]]
STATIC_COLS = [STATIC_COL]
dyn_feature_cols = [c for c in all_feature_cols if c not in STATIC_COLS]
dyn_feature_cols_test = [c for c in test_feature_cols if c not in STATIC_COLS]

print(f"Dynamic features: {len(dyn_feature_cols)}, Static features: {len(STATIC_COLS)}")

train_df = train_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
test_df = test_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

scaler = StandardScaler()
train_dyn_vals = scaler.fit_transform(train_df[dyn_feature_cols])
test_dyn_vals = scaler.transform(test_df[dyn_feature_cols])

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()
train_df_scaled[dyn_feature_cols] = train_dyn_vals
test_df_scaled[dyn_feature_cols] = test_dyn_vals

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(train_df_scaled[TARGET_COL].values)
print("Classes:", list(label_encoder.classes_))

def build_sequences(df_scaled, is_train=True):
    ids = df_scaled[ID_COL].unique()
    ids_sorted = np.sort(ids)
    seq_dyn = []
    seq_static = []
    seq_labels = []
    seq_ids = []

    for sid in ids_sorted:
        sub = df_scaled[df_scaled[ID_COL] == sid].sort_values(TIME_COL)
        dyn = sub[dyn_feature_cols].values
        static_vals = sub[STATIC_COLS].iloc[0].values.astype(float)
        seq_dyn.append(dyn)
        seq_static.append(static_vals)
        seq_ids.append(sid)
        if is_train:
            lbl = sub[TARGET_COL].iloc[0]
            seq_labels.append(lbl)

    seq_dyn = np.array(seq_dyn, dtype=object)
    seq_static = np.stack(seq_static)
    if is_train:
        seq_labels = np.array(label_encoder.transform(seq_labels))
        return seq_dyn, seq_static, seq_labels, np.array(seq_ids)
    else:
        return seq_dyn, seq_static, np.array(seq_ids)

train_dyn_seqs, train_static_seqs, y_seqs, train_ids = build_sequences(train_df_scaled, is_train=True)
test_dyn_seqs, test_static_seqs, test_ids = build_sequences(test_df_scaled, is_train=False)

def create_windows_from_sequences(dyn_seqs, static_seqs, labels, ids, window_size=30, stride=15, is_train=True):
    dyn_windows = []
    static_windows = []
    y_windows = []
    id_windows = []

    for dyn, static_vec, lbl, sid in zip(dyn_seqs, static_seqs,
                                         labels if is_train else [None]*len(dyn_seqs), ids):
        T = dyn.shape[0]
        if T < window_size:
            continue
        for start in range(0, T - window_size + 1, stride):
            end = start + window_size
            dyn_windows.append(dyn[start:end])
            static_windows.append(static_vec)
            id_windows.append(sid)
            if is_train:
                y_windows.append(lbl)

    dyn_windows = np.stack(dyn_windows).astype(np.float32)
    static_windows = np.stack(static_windows).astype(np.float32)
    id_windows = np.array(id_windows)
    if is_train:
        y_windows = np.array(y_windows)
        return dyn_windows, static_windows, y_windows, id_windows
    else:
        return dyn_windows, static_windows, id_windows

WINDOW_SIZE = 24
STRIDE = 12

X_dyn_win, X_static_win, y_win, ids_win = create_windows_from_sequences(
    train_dyn_seqs, train_static_seqs, y_seqs, train_ids,
    window_size=WINDOW_SIZE, stride=STRIDE, is_train=True)

X_dyn_test_win, X_static_test_win, test_ids_win = create_windows_from_sequences(
    test_dyn_seqs, test_static_seqs, labels=None, ids=test_ids,
    window_size=WINDOW_SIZE, stride=STRIDE, is_train=False)

print(f"Train windows: {X_dyn_win.shape}, Test windows: {X_dyn_test_win.shape}")

Dynamic features: 37, Static features: 1
Classes: ['high_pain', 'low_pain', 'no_pain']
Train windows: (7932, 24, 37), Test windows: (15888, 24, 37)


## Class Weights for Imbalanced Data

In [59]:
num_classes = len(np.unique(y_win))
class_counts = np.bincount(y_win, minlength=num_classes)
print("Class counts:", class_counts)

beta = 0.999
effective_num = 1.0 - np.power(beta, class_counts)
effective_num = np.maximum(effective_num, 1e-8)
weights = (1.0 - beta) / effective_num
weights = weights / weights.mean()

print("Class weights:", weights)
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

Class counts: [ 672 1128 6132]
Class weights: [1.35495188 0.98037943 0.66466868]


## Dataset and DataLoader

In [60]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, X_dyn, X_static, y=None):
        self.X_dyn = X_dyn
        self.X_static = X_static
        self.y = y

    def __len__(self):
        return len(self.X_dyn)

    def __getitem__(self, idx):
        dyn_np = np.asarray(self.X_dyn[idx], dtype=np.float32)
        stat_np = np.asarray(self.X_static[idx], dtype=np.float32)
        dyn = torch.from_numpy(dyn_np)
        stat = torch.from_numpy(stat_np)
        if self.y is not None:
            label = torch.tensor(self.y[idx], dtype=torch.long)
            return dyn, stat, label
        else:
            return dyn, stat, torch.tensor(-1, dtype=torch.long)

def make_loader(dataset, batch_size, shuffle):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## Simple Feedforward Neural Network

In [61]:
class SimpleFeedForwardNet(nn.Module):
    def __init__(self, input_size, static_dim, num_classes, hidden_sizes=[256, 128], dropout=0.3):
        super().__init__()
        # Flatten the window: (window_size, features) -> (window_size * features)
        self.flatten_size = input_size
        
        layers = []
        prev_size = self.flatten_size + static_dim
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x_dyn, x_static):
        # x_dyn: (B, window_size, features)
        # x_static: (B, static_dim)
        batch_size = x_dyn.size(0)
        x_flat = x_dyn.view(batch_size, -1)  # Flatten window
        x = torch.cat([x_flat, x_static], dim=1)
        return self.network(x)

## Training Functions

In [62]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for xb_dyn, xb_static, yb in loader:
        xb_dyn = xb_dyn.to(device)
        xb_static = xb_static.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb_dyn, xb_static)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    f1_micro = f1_score(all_labels, all_preds, average='micro')
    return avg_loss, f1_weighted, f1_micro

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb_dyn, xb_static, yb in loader:
            xb_dyn = xb_dyn.to(device)
            xb_static = xb_static.to(device)
            yb = yb.to(device)

            logits = model(xb_dyn, xb_static)
            loss = criterion(logits, yb)
            total_loss += loss.item() * yb.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    f1_micro = f1_score(all_labels, all_preds, average='micro')
    return avg_loss, f1_weighted, f1_micro

def train_model(model, train_loader, val_loader, criterion, learning_rate, epochs, patience, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    best_val_f1_weighted = float("-inf")
    best_val_f1_micro = 0.0
    best_state_dict = None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_f1_weighted, train_f1_micro = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_f1_weighted, val_f1_micro = validate(model, val_loader, criterion, device)
        scheduler.step(val_f1_weighted)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | train_loss={train_loss:.4f} f1_w={train_f1_weighted:.4f} f1_m={train_f1_micro:.4f} | "
                  f"val_loss={val_loss:.4f} f1_w={val_f1_weighted:.4f} f1_m={val_f1_micro:.4f}")

        if val_f1_weighted > best_val_f1_weighted:
            best_val_f1_weighted = val_f1_weighted
            best_val_f1_micro = val_f1_micro
            best_state_dict = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break
    return best_state_dict, best_val_f1_weighted, best_val_f1_micro


## K-Fold Cross Validation

In [63]:
K_FOLDS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 200
PATIENCE = 30
HIDDEN_SIZES = [512, 256, 128, 64]
DROPOUT = 0.4

input_size = X_dyn_win.shape[1] * X_dyn_win.shape[2]  # window_size * num_features
static_dim = X_static_win.shape[1]

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)

sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_f1_weighted_scores = []
fold_f1_micro_scores = []
model_paths = []

os.makedirs("models", exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_dyn_win, y_win, groups=ids_win), 1):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/{K_FOLDS}")
    print(f"{'='*60}")
    
    X_tr_dyn, X_val_dyn = X_dyn_win[train_idx], X_dyn_win[val_idx]
    X_tr_stat, X_val_stat = X_static_win[train_idx], X_static_win[val_idx]
    y_tr, y_val = y_win[train_idx], y_win[val_idx]

    train_ds = TimeSeriesWindowDataset(X_tr_dyn, X_tr_stat, y_tr)
    val_ds = TimeSeriesWindowDataset(X_val_dyn, X_val_stat, y_val)

    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

    model = SimpleFeedForwardNet(
        input_size=input_size,
        static_dim=static_dim,
        num_classes=num_classes,
        hidden_sizes=HIDDEN_SIZES,
        dropout=DROPOUT
    ).to(DEVICE)

    best_state_dict, best_val_f1_weighted, best_val_f1_micro = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        patience=PATIENCE,
        device=DEVICE
    )

    fold_f1_weighted_scores.append(best_val_f1_weighted)
    fold_f1_micro_scores.append(best_val_f1_micro)
    
    fold_path = f"models/fold_{fold}_best.pt"
    torch.save(best_state_dict, fold_path)
    model_paths.append(fold_path)
    print(f"  Best validation F1 weighted: {best_val_f1_weighted:.4f}, F1 micro: {best_val_f1_micro:.4f}")

print(f"\n{'='*60}")
print("K-Fold Cross Validation Results")
print(f"{'='*60}")
print(f"F1 Weighted - Mean: {np.mean(fold_f1_weighted_scores):.4f} ± {np.std(fold_f1_weighted_scores):.4f}")
print(f"  Individual fold scores: {[f'{f:.4f}' for f in fold_f1_weighted_scores]}")
print(f"F1 Micro - Mean: {np.mean(fold_f1_micro_scores):.4f} ± {np.std(fold_f1_micro_scores):.4f}")
print(f"  Individual fold scores: {[f'{f:.4f}' for f in fold_f1_micro_scores]}")


Fold 1/5
  Epoch 001 | train_loss=0.9389 f1_w=0.7465 f1_m=0.7459 | val_loss=0.7441 f1_w=0.9062 f1_m=0.9085
  Epoch 010 | train_loss=0.6079 f1_w=0.9608 f1_m=0.9605 | val_loss=0.6781 f1_w=0.9252 f1_m=0.9223
  Epoch 020 | train_loss=0.5731 f1_w=0.9776 f1_m=0.9774 | val_loss=0.6743 f1_w=0.9254 f1_m=0.9248
  Epoch 030 | train_loss=0.5477 f1_w=0.9912 f1_m=0.9912 | val_loss=0.6853 f1_w=0.9275 f1_m=0.9248
  Epoch 040 | train_loss=0.5385 f1_w=0.9954 f1_m=0.9954 | val_loss=0.6837 f1_w=0.9295 f1_m=0.9273
  Epoch 050 | train_loss=0.5356 f1_w=0.9962 f1_m=0.9962 | val_loss=0.6615 f1_w=0.9405 f1_m=0.9380
  Epoch 060 | train_loss=0.5364 f1_w=0.9951 f1_m=0.9951 | val_loss=0.6764 f1_w=0.9306 f1_m=0.9298
  Epoch 070 | train_loss=0.5320 f1_w=0.9980 f1_m=0.9979 | val_loss=0.6765 f1_w=0.9303 f1_m=0.9292
  Epoch 080 | train_loss=0.5302 f1_w=0.9986 f1_m=0.9986 | val_loss=0.6755 f1_w=0.9327 f1_m=0.9311
  Early stopping at epoch 80
  Best validation F1 weighted: 0.9405, F1 micro: 0.9380

Fold 2/5
  Epoch 001 |

## Generate Test Predictions Using Last Window

In [64]:
def get_last_window_indices(window_ids):
    """Get indices of the last window for each unique sample."""
    unique_ids = np.unique(window_ids)
    last_window_indices = []
    
    for sid in unique_ids:
        mask = window_ids == sid
        indices = np.where(mask)[0]
        last_window_indices.append(indices[-1])  # Last window for this sample
    
    return np.array(last_window_indices), unique_ids

def predict_last_window(model_paths, X_dyn, X_static, window_ids, device, batch_size=128):
    """Predict using ensemble of models on last window only."""
    last_indices, sample_ids = get_last_window_indices(window_ids)
    
    # Get last window data
    X_dyn_last = X_dyn[last_indices]
    X_static_last = X_static[last_indices]
    
    test_ds = TimeSeriesWindowDataset(X_dyn_last, X_static_last, y=None)
    test_loader = make_loader(test_ds, batch_size=batch_size, shuffle=False)
    
    all_fold_probs = []
    
    for path in model_paths:
        model = SimpleFeedForwardNet(
            input_size=input_size,
            static_dim=static_dim,
            num_classes=num_classes,
            hidden_sizes=HIDDEN_SIZES,
            dropout=DROPOUT
        ).to(device)
        
        state_dict = torch.load(path, map_location=device)
        model.load_state_dict(state_dict)
        model.eval()
        
        fold_probs = []
        with torch.no_grad():
            for xb_dyn, xb_static, _ in test_loader:
                xb_dyn = xb_dyn.to(device)
                xb_static = xb_static.to(device)
                logits = model(xb_dyn, xb_static)
                probs = F.softmax(logits, dim=1)
                fold_probs.append(probs.cpu().numpy())
        
        fold_probs = np.concatenate(fold_probs, axis=0)
        all_fold_probs.append(fold_probs)
    
    # Average probabilities across folds
    mean_probs = np.mean(all_fold_probs, axis=0)
    predictions = mean_probs.argmax(axis=1)
    
    return sample_ids, predictions

print("Generating predictions using last window from each sample...")
sample_ids_ordered, preds = predict_last_window(
    model_paths=model_paths,
    X_dyn=X_dyn_test_win,
    X_static=X_static_test_win,
    window_ids=test_ids_win,
    device=DEVICE,
    batch_size=BATCH_SIZE * 2
)

preds_labels = label_encoder.inverse_transform(preds)

Generating predictions using last window from each sample...


## Create Submission File

In [65]:
submission = pd.DataFrame({
    ID_COL: sample_ids_ordered,
    TARGET_COL: preds_labels
})
submission = submission.sort_values(ID_COL).reset_index(drop=True)
submission.to_csv("submission_simple_ffnn.csv", index=False)

print("\n" + "="*60)
print("SUBMISSION FILE CREATED!")
print("="*60)
print(submission.head(10))
print(f"\nTotal samples: {len(submission)}")
print(f"\nClass distribution:")
print(submission[TARGET_COL].value_counts())


SUBMISSION FILE CREATED!
   sample_index    label
0             0  no_pain
1             1  no_pain
2             2  no_pain
3             3  no_pain
4             4  no_pain
5             5  no_pain
6             6  no_pain
7             7  no_pain
8             8  no_pain
9             9  no_pain

Total samples: 1324

Class distribution:
label
no_pain      1039
low_pain      178
high_pain     107
Name: count, dtype: int64
